# 生产级记忆模式

> **让智能体记忆在 demo 里跑通并不难。真正难的是规模化之后还能保持快速、便宜、隐私且可靠。生产级记忆模式弥合「在我笔记本上能跑」和「能服务一百万用户」之间的差距。**

大多数智能体记忆教程只讲到「把嵌入存进向量数据库」。但在生产环境里，你会遇到它们从不讨论的问题：记忆库涨到 50 GB 时怎么办？有些用户存了一万条记忆，怎么把检索控制在 100 ms 以内？GDPR（欧盟隐私法）怎么应对：你能删掉一个用户在嵌入、图谱和缓存里的所有痕迹吗？一个失控的智能体一夜烧掉 500 美元嵌入调用费，怎么拦住？

可以把这件事想象成经营餐厅后厨和在家做饭的区别。在家里你有一口锅、一台冰箱。在餐厅里，你需要热灶台（马上要出的菜）、冷藏间（当天的食材）和深冷柜（长期存放的库存）。你还需要卫生检查、成本控制，以及停电时的应急预案。

本 notebook 实现了一套**可部署到生产的记忆基础设施**，覆盖六大支柱：

1. **分层存储：** 热（内存 / Redis）、温（Postgres + pgvector）、冷（压缩归档）三层，根据访问模式自动升级和降级。
2. **图关系：** 承载结构化知识的实体关系层，与向量相似度互补。
3. **隐私与个人身份信息（PII）处理：** 检测、脱敏，以及跨所有层的完整用户删除（「被遗忘权」）。
4. **记忆预算限制：** 对每用户的 token 数、记忆条数和存储量设硬上限，防止无限增长。
5. **成本控制：** 跟踪并封顶嵌入、LLM 抽取和存储的支出。
6. **可观测性：** 用结构化指标跟踪延迟、缓存命中率、层间分布和成本。

**读完本 notebook 你将拥有：**
- 一套能运行的热 / 温 / 冷三层记忆系统。
- 带完整用户数据删除能力的 PII 检测与脱敏。
- 能防止记忆无限增长的预算限制。
- 支出上限可配置的成本跟踪。
- 一个清晰的生产级记忆架构心智模型。


## 核心概念

- **热 / 温 / 冷三层：** 一种存储层级。可以想象成一个厨房：热层是台面（即取即用，空间有限），温层是冰箱（取用很快，容量更大），冷层是地下室的冷柜（取用慢，但容量巨大且便宜）。频繁访问的记忆放在快但贵的存储里，很少访问的则进入压缩的冷归档。
- **用 Redis 存短期记忆：** Redis 是一种内存数据存储，读取速度达到亚毫秒级。你用它存放最近的上下文、会话状态和需要频繁检索的用户事实。基于 TTL 的过期机制（到期自动删除）让它保持精简。
- **用 Postgres + pgvector 存长期记忆：** 一个带向量相似度搜索的关系型数据库。它把 Postgres 的可靠性和查询灵活性与基于嵌入的检索结合起来。对大多数生产系统来说，这是承担主要负载的一层。
- **用图数据库存关系：** 实体关系存储（如 Neo4j，或进程内的轻量图），用来承载结构化知识。例如「Alice works_at Anthropic」。它支持图遍历查询，这类问题向量相似度回答不了。
- **隐私与 PII 处理：** 在入库前检测记忆中的 PII（姓名、邮箱、社会安全号），对敏感字段脱敏或加密，并实现跨所有层的完整用户数据删除，以满足 GDPR/CCPA 的合规要求。
- **记忆预算限制：** 对每用户的记忆条数、token 总量和存储字节数设硬上限。超限时由淘汰策略自动修剪低价值记忆：LRU 指「最久未用的先删」，重要性加权指「最不重要的先删」。
- **成本控制：** 跟踪每次记忆操作（嵌入生成、LLM 抽取、存储）的美元成本，并按用户、按天和全局设支出上限，防止账单失控。


## Architecture

<p align="center">
  <img src="../../images/diagrams/30_production_memory_patterns.svg" alt="架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart TD
    subgraph Ingestion["Memory Ingestion"]
        MSG["New Message"] --> PII["PII Scanner\n& Redactor"]
        PII --> EXT["Fact Extractor\n(LLM)"]
        EXT --> EMB["Embedding\nGenerator"]
        EMB --> COST["Cost Tracker\n$ gate"]
    end

    subgraph Tiers["Tiered Storage"]
        direction LR
        HOT["Hot Tier\n─────────\nIn-Memory / Redis\nTTL: minutes-hours\nLatency: <1ms"]
        WARM["Warm Tier\n─────────\nPostgres + pgvector\nTTL: days-months\nLatency: 5-50ms"]
        COLD["Cold Tier\n─────────\nCompressed Archive\nTTL: indefinite\nLatency: 100ms+"]
    end

    subgraph Graph["Relationship Layer"]
        GDB[("Entity Graph\n─────────\nnodes / edges\ntraversal queries")]
    end

    subgraph Controls["Guardrails"]
        BUD["Budget Enforcer\nmax tokens / max count"]
        PRV["Privacy Manager\nGDPR delete / audit log"]
        OBS["Observability\nmetrics / alerts"]
    end

    COST --> HOT
    HOT -->|"demote\n(TTL expiry)"| WARM
    WARM -->|"demote\n(low access)"| COLD
    COLD -->|"promote\n(re-accessed)"| WARM
    WARM -->|"promote\n(hot query)"| HOT
    EXT --> GDB

    HOT & WARM & COLD --> BUD
    HOT & WARM & COLD --> PRV
    HOT & WARM & COLD --> OBS

    style HOT fill:#ef4444,color:#fff
    style WARM fill:#f59e0b,color:#fff
    style COLD fill:#3b82f6,color:#fff
    style GDB fill:#8b5cf6,color:#fff
    style PII fill:#059669,color:#fff
    style COST fill:#d97706,color:#fff
```

</details>


In [ ]:
# 安装所需的包（只需运行一次）
%pip install -q python-dotenv numpy deepagents langchain-openai

我们导入所需的库，并初始化 Anthropic 客户端。你需要在 `.env` 文件里配置 `LLM_API_KEY`。


In [ ]:
import os
import json
import re
import time
import zlib
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from typing import Any

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 核心实现

### 数据模型与层定义

每条记忆都带有元数据，供分层系统决定放在哪一层：访问次数、最近访问时间、重要性分数和当前所在层。系统会根据访问模式自动升级和降级记录。


In [ ]:
class MemoryTier(Enum):
    """记忆记录的存储层。"""
    HOT = "hot"      # 内存 / Redis：亚毫秒级读取
    WARM = "warm"    # Postgres + pgvector：毫秒级读取
    COLD = "cold"    # 压缩归档：100ms 以上读取


@dataclass
class MemoryRecord:
    """带生产级元数据的单条记忆。"""
    id: str
    user_id: str
    content: str
    embedding: list[float] | None = None
    created_at: datetime = field(default_factory=datetime.utcnow)
    last_accessed: datetime = field(default_factory=datetime.utcnow)
    access_count: int = 0
    importance: float = 0.5         # 0.0 = 无关紧要，1.0 = 至关重要
    tier: MemoryTier = MemoryTier.WARM
    ttl_seconds: int | None = None  # None 表示永不过期
    tags: list[str] = field(default_factory=list)
    pii_redacted: bool = False
    token_estimate: int = 0         # 内容的近似 token 数
    compressed_bytes: bytes | None = None  # 供冷层使用

    def is_expired(self) -> bool:
        if self.ttl_seconds is None:
            return False
        age = (datetime.utcnow() - self.created_at).total_seconds()
        return age > self.ttl_seconds

    def touch(self):
        """记录一次访问。"""
        self.last_accessed = datetime.utcnow()
        self.access_count += 1

    def __repr__(self) -> str:
        short = self.content[:45] + "..." if len(self.content) > 45 else self.content
        return f"Memory({self.id}, tier={self.tier.value}, accesses={self.access_count}: {short})"


def estimate_tokens(text: str) -> int:
    """粗略估算 token 数：英文约 4 个字符折 1 个 token。"""
    return max(1, len(text) // 4)


print("✓ MemoryTier enum and MemoryRecord dataclass defined")

### 热层：带 TTL 的内存缓存

热层模拟了一个类似 Redis 的缓存：亚毫秒级读取、TTL 到期自动删除（记录过了存活时间就消失），以及容量不足时向温层的 LRU 淘汰（最久未使用的记录先被移出）。

生产环境中你会用真正的 Redis。这里我们用 Python 实现同样的语义，让 notebook 保持自包含。

**关键设计决策：**
- 每用户固定容量（可配置），防止单个用户占满缓存。
- TTL 过期在每次读取时检查，不需要后台线程。
- 达到容量上限时按 LRU 淘汰：最久未访问的记录降级到温层。


In [ ]:
class HotTierCache:
    """模拟 Redis 的内存缓存，带 TTL 和 LRU 淘汰。

    生产环境中请替换为 redis-py：
        import redis
        r = redis.Redis(host='localhost', port=6379)
        r.setex(key, ttl, json.dumps(record))
    """

    def __init__(self, max_per_user: int = 50, default_ttl: int = 3600):
        self.store: dict[str, MemoryRecord] = {}  # id -> 记录
        self.user_index: dict[str, set[str]] = defaultdict(set)  # user_id -> {id 集合}
        self.max_per_user = max_per_user
        self.default_ttl = default_ttl
        self.stats = {"hits": 0, "misses": 0, "evictions": 0, "expirations": 0}

    def put(self, record: MemoryRecord) -> MemoryRecord | None:
        """缓存一条记录；容量超限时返回被淘汰的记录。"""
        record.tier = MemoryTier.HOT
        if record.ttl_seconds is None:
            record.ttl_seconds = self.default_ttl

        evicted = None
        user_ids = self.user_index[record.user_id]

        # 达到容量上限时按 LRU 淘汰
        if len(user_ids) >= self.max_per_user and record.id not in user_ids:
            lru_id = min(user_ids, key=lambda rid: self.store[rid].last_accessed)
            evicted = self.store.pop(lru_id)
            user_ids.discard(lru_id)
            evicted.tier = MemoryTier.WARM  # 降级
            self.stats["evictions"] += 1

        self.store[record.id] = record
        user_ids.add(record.id)
        return evicted

我们为 `HotTierCache` 补上其余方法。`get` 方法在返回记录前检查是否过期。`get_user_records` 在读取时顺便清除过期条目。`delete_user` 一次性删除某用户的所有缓存记录。


In [ ]:
def get(self, record_id: str) -> MemoryRecord | None:
    """读取一条记录；未命中或已过期时返回 None。"""
    record = self.store.get(record_id)
    if record is None:
        self.stats["misses"] += 1
        return None
    if record.is_expired():
        self._remove(record)
        self.stats["expirations"] += 1
        self.stats["misses"] += 1
        return None
    record.touch()
    self.stats["hits"] += 1
    return record

def _remove(self, record: MemoryRecord):
    self.store.pop(record.id, None)
    self.user_index[record.user_id].discard(record.id)

def get_user_records(self, user_id: str) -> list[MemoryRecord]:
    """获取某用户的所有缓存记录（顺便清除过期条目）。"""
    ids = list(self.user_index.get(user_id, set()))
    records = []
    for rid in ids:
        r = self.store.get(rid)
        if r and not r.is_expired():
            records.append(r)
        elif r:
            self._remove(r)
            self.stats["expirations"] += 1
    return records

def delete_user(self, user_id: str) -> int:
    """删除某用户的全部记录，返回删除数量。"""
    ids = list(self.user_index.pop(user_id, set()))
    for rid in ids:
        self.store.pop(rid, None)
    return len(ids)

@property
def hit_rate(self) -> float:
    total = self.stats["hits"] + self.stats["misses"]
    return self.stats["hits"] / total if total > 0 else 0.0

def __len__(self):
    return len(self.store)

HotTierCache.get = get
HotTierCache._remove = _remove
HotTierCache.get_user_records = get_user_records
HotTierCache.delete_user = delete_user
HotTierCache.hit_rate = hit_rate
HotTierCache.__len__ = __len__

print("✓ HotTierCache defined (Redis-like in-memory cache with TTL + LRU)")

### 温层：向量库（Postgres + pgvector）

温层模拟了带 pgvector 扩展的 Postgres：持久化存储，支持基于嵌入的相似度搜索。这是承担主要负载的一层，大多数记忆都存放在这里。

**生产环境中**你会用真正的 pgvector：
```sql
CREATE TABLE memories (
    id UUID PRIMARY KEY,
    user_id UUID NOT NULL,
    content TEXT,
    embedding vector(1536),
    created_at TIMESTAMPTZ,
    importance FLOAT
);
CREATE INDEX ON memories USING ivfflat (embedding vector_cosine_ops);
```

这里我们用 NumPy 实现同样的检索语义，计算余弦相似度（衡量两个向量的相似程度，取值范围 -1 到 1）。


In [ ]:
class WarmTierStore:
    """用余弦相似度搜索模拟 Postgres + pgvector。

    生产环境中请替换为：
        import psycopg2
        cur.execute(
            "SELECT * FROM memories WHERE user_id = %s "
            "ORDER BY embedding <=> %s::vector LIMIT %s",
            (user_id, query_embedding, k)
        )
    """

    def __init__(self):
        self.records: dict[str, MemoryRecord] = {}
        self.user_index: dict[str, set[str]] = defaultdict(set)

    def put(self, record: MemoryRecord) -> None:
        record.tier = MemoryTier.WARM
        record.compressed_bytes = None  # 从冷层升级后不再保留压缩数据
        self.records[record.id] = record
        self.user_index[record.user_id].add(record.id)

    def get(self, record_id: str) -> MemoryRecord | None:
        record = self.records.get(record_id)
        if record:
            record.touch()
        return record

现在我们为温层添加搜索和日常维护方法。`search` 方法计算查询嵌入与已存嵌入之间的余弦相似度（衡量两个向量的相似程度）。`get_low_access_records` 找出许久未被访问的陈旧记录，它们是降级到冷存储的候选。


In [ ]:
def search(self, user_id: str, query_embedding: list[float], k: int = 5) -> list[MemoryRecord]:
    """在用户记忆上做余弦相似度搜索。"""
    user_ids = self.user_index.get(user_id, set())
    if not user_ids:
        return []

    candidates = []
    q = np.array(query_embedding)
    q_norm = np.linalg.norm(q)
    if q_norm == 0:
        return []

    for rid in user_ids:
        record = self.records[rid]
        if record.embedding is None:
            continue
        e = np.array(record.embedding)
        e_norm = np.linalg.norm(e)
        if e_norm == 0:
            continue
        sim = float(np.dot(q, e) / (q_norm * e_norm))
        candidates.append((sim, record))

    candidates.sort(key=lambda x: x[0], reverse=True)
    results = [r for _, r in candidates[:k]]
    for r in results:
        r.touch()
    return results

WarmTierStore.search = search

这些小方法处理用户级操作。`get_user_records` 返回某用户存储的全部记忆。`get_low_access_records` 找出最近 N 天未被访问的记录。这些陈旧记录是降级到冷存储的候选。


In [ ]:
def get_user_records(self, user_id: str) -> list[MemoryRecord]:
    return [self.records[rid] for rid in self.user_index.get(user_id, set())]

def delete_user(self, user_id: str) -> int:
    ids = list(self.user_index.pop(user_id, set()))
    for rid in ids:
        self.records.pop(rid, None)
    return len(ids)

def get_low_access_records(self, user_id: str, threshold_days: int = 30) -> list[MemoryRecord]:
    """找出最近 N 天未被访问的记录（降级到冷层的候选）。"""
    cutoff = datetime.utcnow() - timedelta(days=threshold_days)
    return [
        self.records[rid]
        for rid in self.user_index.get(user_id, set())
        if self.records[rid].last_accessed < cutoff
    ]

def remove(self, record_id: str) -> MemoryRecord | None:
    record = self.records.pop(record_id, None)
    if record:
        self.user_index[record.user_id].discard(record_id)
    return record

def __len__(self):
    return len(self.records)

WarmTierStore.get_user_records = get_user_records
WarmTierStore.delete_user = delete_user
WarmTierStore.get_low_access_records = get_low_access_records
WarmTierStore.remove = remove
WarmTierStore.__len__ = __len__

print("✓ WarmTierStore defined (pgvector-like vector store with cosine similarity)")

### 冷层：压缩归档

冷层以压缩形式存放很少被访问的记忆。生产环境中可以是 S3 + Glacier（Amazon 的廉价归档存储），也可以是独立 Postgres 表中的压缩列。读取慢一些，但存储成本极低。

**降级标准：** 超过 30 天无访问且重要性分数低。
**升级：** 冷层记忆一旦被访问，就会解压并移回温层。


In [ ]:
class ColdTierArchive:
    """压缩冷存储，存放很少被访问的记忆。

    生产环境中可替换为：
        - S3 with lifecycle policies (S3 -> Glacier after 90 days)
        - Postgres partitioned table with TOAST compression
        - Separate cold-storage database
    """

    def __init__(self):
        self.records: dict[str, MemoryRecord] = {}
        self.user_index: dict[str, set[str]] = defaultdict(set)
        self.bytes_saved: int = 0

    def archive(self, record: MemoryRecord) -> None:
        """压缩并存储一条记忆记录。"""
        record.tier = MemoryTier.COLD
        original_size = len(record.content.encode("utf-8"))
        record.compressed_bytes = zlib.compress(record.content.encode("utf-8"))
        record.embedding = None  # 丢弃嵌入以节省空间
        compressed_size = len(record.compressed_bytes)
        self.bytes_saved += (original_size - compressed_size)

        self.records[record.id] = record
        self.user_index[record.user_id].add(record.id)

    def retrieve(self, record_id: str) -> MemoryRecord | None:
        """解压并返回冷层记录（供升级回温层）。"""
        record = self.records.get(record_id)
        if record and record.compressed_bytes:
            record.content = zlib.decompress(record.compressed_bytes).decode("utf-8")
            record.touch()
        return record

    def get_user_records(self, user_id: str) -> list[MemoryRecord]:
        return [self.records[rid] for rid in self.user_index.get(user_id, set())]

    def delete_user(self, user_id: str) -> int:
        ids = list(self.user_index.pop(user_id, set()))
        for rid in ids:
            self.records.pop(rid, None)
        return len(ids)

    def remove(self, record_id: str) -> MemoryRecord | None:
        return self.records.pop(record_id, None)

    def __len__(self):
        return len(self.records)


print("✓ ColdTierArchive defined (zlib-compressed cold storage)")

### 图关系存储

向量相似度适合「找出与 X 相关的记忆」。但它回答不了结构化问题，比如「Alice 和谁共事？」或「用户住过哪些地方？」一个轻量的实体关系图可以与向量搜索互补，覆盖这类场景。

可以把它想象成社交网络地图。向量搜索找出*听起来相似*的东西，图搜索找出*相互关联*的东西。两者都有用，回答的是不同类型的问题。

**生产环境中**你会用 Neo4j、Amazon Neptune，甚至带递归 CTE 的 Postgres（一种用于遍历关系的 SQL 特性）。这里我们用邻接表图（一个字典，把每个实体映射到它的连接）。


In [ ]:
@dataclass
class GraphEdge:
    """两个实体之间的关系。"""
    source: str         # 例如 "Alice"
    relation: str       # 例如 "works_at"
    target: str         # 例如 "Anthropic"
    user_id: str
    timestamp: datetime = field(default_factory=datetime.utcnow)
    memory_id: str | None = None  # 回链到来源记忆

`GraphRelationshipStore` 是一个轻量的内存图。它用邻接表（一个字典，键是实体，值是它的边列表）把实体名映射到各自的连接。`get_neighbors` 方法用 BFS（广度优先搜索）找出 N 跳以内的所有实体。


In [ ]:
class GraphRelationshipStore:
    """轻量级实体关系图。

    生产环境中请替换为：
        from neo4j import GraphDatabase
        driver = GraphDatabase.driver(uri, auth=(user, password))
        session.run(
            "MERGE (a:Entity {name: $source}) "
            "MERGE (b:Entity {name: $target}) "
            "CREATE (a)-[:REL {type: $rel}]->(b)",
            source=edge.source, target=edge.target, rel=edge.relation
        )
    """

    def __init__(self):
        self.edges: list[GraphEdge] = []
        self.adjacency: dict[str, list[GraphEdge]] = defaultdict(list)

    def add_edge(self, edge: GraphEdge) -> None:
        self.edges.append(edge)
        self.adjacency[edge.source.lower()].append(edge)

    def query(self, entity: str, relation: str | None = None) -> list[GraphEdge]:
        """查找某实体的边，可按关系类型过滤。"""
        edges = self.adjacency.get(entity.lower(), [])
        if relation:
            edges = [e for e in edges if e.relation == relation]
        return edges

    def get_neighbors(self, entity: str, depth: int = 1) -> set[str]:
        """BFS 遍历，找出 N 跳以内的实体。"""
        visited = set()
        frontier = {entity.lower()}
        for _ in range(depth):
            next_frontier = set()
            for node in frontier:
                if node in visited:
                    continue
                visited.add(node)
                for edge in self.adjacency.get(node, []):
                    next_frontier.add(edge.target.lower())
            frontier = next_frontier - visited
        return visited | frontier

    def delete_user(self, user_id: str) -> int:
        """删除某用户的全部边。"""
        original = len(self.edges)
        self.edges = [e for e in self.edges if e.user_id != user_id]
        # 重建邻接表
        self.adjacency.clear()
        for edge in self.edges:
            self.adjacency[edge.source.lower()].append(edge)
        return original - len(self.edges)

    def __len__(self):
        return len(self.edges)


print("✓ GraphRelationshipStore defined (adjacency-list entity graph)")

### 隐私与 PII 处理

生产级记忆系统必须小心处理 PII。PII 包括邮箱地址、电话号码、社会安全号、银行卡号等。本模块提供：

1. **检测：** 用正则扫描常见 PII 模式（邮箱、电话、社会安全号）。
2. **脱敏：** 入库前把检测到的 PII 替换成占位符。
3. **被遗忘权：** 跨所有层和图存储完整删除用户全部数据。
4. **审计日志：** 记录每一次删除，留作合规凭证。

> **注意：** 生产环境请使用专门的 PII 检测服务（如 AWS Comprehend、Google DLP 或 Microsoft Presidio），不要只靠正则。正则能抓住常见格式，但抓不住依赖上下文的 PII，比如姓名和地址。


In [ ]:
class PIIHandler:
    """检测、脱敏并管理记忆内容中的 PII。

    生产环境中可考虑：
        - Microsoft Presidio (open-source, pluggable)
        - AWS Comprehend PII detection
        - Google Cloud DLP
        - Custom NER models for domain-specific PII
    """

    # 常见 PII 模式（以美国格式为主；其他地区请扩展）
    PII_PATTERNS = {
        "email": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"),
        "phone_us": re.compile(r"\b(?:\+1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"),
        "ssn": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
        "credit_card": re.compile(r"\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b"),
        "ip_address": re.compile(r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b"),
    }

    def __init__(self):
        self.audit_log: list[dict] = []

    def scan(self, text: str) -> dict[str, list[str]]:
        """扫描文本中的 PII，返回 {pii 类型: [匹配值]}。"""
        found = {}
        for pii_type, pattern in self.PII_PATTERNS.items():
            matches = pattern.findall(text)
            if matches:
                found[pii_type] = matches
        return found

    def redact(self, text: str) -> tuple[str, dict[str, list[str]]]:
        """脱敏文本中的 PII，返回 (脱敏后文本, 检测到的 PII)。"""
        detected = self.scan(text)
        redacted = text
        for pii_type, values in detected.items():
            for value in values:
                placeholder = f"[REDACTED_{pii_type.upper()}]"
                redacted = redacted.replace(value, placeholder)
        return redacted, detected

    def log_deletion(self, user_id: str, tier: str, count: int):
        """Record a deletion event for compliance audit."""
        self.audit_log.append({
            "action": "user_data_deletion",
            "user_id": user_id,
            "tier": tier,
            "records_deleted": count,
            "timestamp": datetime.utcnow().isoformat(),
        })

我们来测试 PII 扫描器。跑三段示例文本：前两段包含邮箱、电话号码、社会安全号和银行卡号，第三段不含 PII。观察脱敏器如何把每个检测到的值替换成带标签的占位符。


In [ ]:
# ── PII 检测演示 ──
pii = PIIHandler()

test_texts = [
    "My email is alice@example.com and my phone is (555) 123-4567.",
    "SSN: 123-45-6789, credit card 4111-1111-1111-1111.",
    "Alice works at Anthropic as a senior engineer.",  # 不含 PII
]

print("\u2713 PIIHandler defined\n")
print("\u2500\u2500 PII Detection Demo \u2500\u2500")
for text in test_texts:
    redacted, found = pii.redact(text)
    if found:
        print(f"  Original:  {text}")
        print(f"  Redacted:  {redacted}")
        print(f"  Found PII: {found}\n")
    else:
        print(f"  Clean:     {text}\n")

### 记忆预算限制

没有硬上限，记忆库就会无限膨胀。一个重度用户可能存下 10 万条记忆，把所有人的查询延迟都拖慢。预算限制为每个用户封顶增长，超限时淘汰低价值记忆。

**三个预算维度：**
1. **最大记忆条数：** 每用户记录数的硬上限。
2. **最大 token 数：** 所有记忆的 token 总预算（控制它们占用多少上下文窗口）。
3. **淘汰策略：** 超预算时按 `importance x recency` 给每条记忆打分，淘汰得分最低的那些。


In [ ]:
class MemoryBudgetEnforcer:
    """用重要性加权淘汰执行每用户记忆限额。

    每次写入都会检查预算。超限时返回价值最低的记忆，
    供降级或删除。
    """

    def __init__(
        self,
        max_memories_per_user: int = 500,
        max_tokens_per_user: int = 100_000,
    ):
        self.max_memories = max_memories_per_user
        self.max_tokens = max_tokens_per_user

    def check_budget(self, records: list[MemoryRecord]) -> dict:
        """检查用户的记忆是否超出预算限制。

        Returns:
            返回包含 'over_count'、'over_tokens'、'eviction_candidates' 的字典
        """
        total_tokens = sum(r.token_estimate for r in records)
        count = len(records)

        over_count = max(0, count - self.max_memories)
        over_tokens = max(0, total_tokens - self.max_tokens)

        # 给每条记录打分：分数越低，越该被淘汰
        scored = []
        now = datetime.utcnow()
        for r in records:
            age_days = max(1, (now - r.last_accessed).total_seconds() / 86400)
            recency = 1.0 / age_days  # 越新近分数越高
            score = r.importance * 0.6 + recency * 0.3 + min(r.access_count / 10, 1.0) * 0.1
            scored.append((score, r))

        scored.sort(key=lambda x: x[0])  # 分数最低的排前面

        eviction_candidates = []
        tokens_to_free = over_tokens
        count_to_free = over_count

        for score, record in scored:
            if count_to_free <= 0 and tokens_to_free <= 0:
                break
            eviction_candidates.append(record)
            count_to_free -= 1
            tokens_to_free -= record.token_estimate

        return {
            "within_budget": over_count <= 0 and over_tokens <= 0,
            "current_count": count,
            "current_tokens": total_tokens,
            "over_count": over_count,
            "over_tokens": over_tokens,
            "eviction_candidates": eviction_candidates,
        }


print("✓ MemoryBudgetEnforcer defined")

### 成本控制与跟踪

每次记忆操作都有美元成本：嵌入生成、LLM 事实抽取、存储读写。没有跟踪，成本会悄悄爬升，直到月底账单送到眼前。成本跟踪器记录每一次操作，并执行可配置的支出上限。


In [ ]:
class CostTracker:
    """跟踪并限制记忆操作的美元成本。

    费率为近似值且可配置；模型变化时请更新单价。
    """

    # 近似单价（美元），请按你的模型更新
    DEFAULT_RATES = {
        "embedding_per_1k_tokens": 0.00002,     # text-embedding-3-small
        "extraction_input_per_1k": 0.003,        # Claude Sonnet input
        "extraction_output_per_1k": 0.015,       # Claude Sonnet output
        "storage_per_gb_month": 0.25,            # pgvector on managed Postgres
    }

    def __init__(
        self,
        daily_limit: float = 10.0,
        monthly_limit: float = 200.0,
        rates: dict[str, float] | None = None,
    ):
        self.daily_limit = daily_limit
        self.monthly_limit = monthly_limit
        self.rates = rates or self.DEFAULT_RATES
        self.ledger: list[dict] = []

    def record(self, operation: str, tokens: int = 0, details: str = "") -> float:
        """记录一次成本事件，返回该次成本（美元）。"""
        if operation == "embedding":
            cost = (tokens / 1000) * self.rates["embedding_per_1k_tokens"]
        elif operation == "extraction_input":
            cost = (tokens / 1000) * self.rates["extraction_input_per_1k"]
        elif operation == "extraction_output":
            cost = (tokens / 1000) * self.rates["extraction_output_per_1k"]
        else:
            cost = 0.0

        self.ledger.append({
            "operation": operation,
            "tokens": tokens,
            "cost_usd": cost,
            "timestamp": datetime.utcnow().isoformat(),
            "details": details,
        })
        return cost

我们为成本跟踪器添加报表方法。`get_daily_spend` 汇总今天的花费。`check_limits` 把当前支出与每日和每月上限比较。`get_breakdown` 按操作类型分组成本，让你看清钱花在了哪里。


In [ ]:
def get_daily_spend(self) -> float:
    today = datetime.utcnow().date()
    return sum(
        e["cost_usd"] for e in self.ledger
        if datetime.fromisoformat(e["timestamp"]).date() == today
    )

def get_total_spend(self) -> float:
    return sum(e["cost_usd"] for e in self.ledger)

def check_limits(self) -> dict:
    daily = self.get_daily_spend()
    total = self.get_total_spend()
    return {
        "daily_spend": daily,
        "daily_limit": self.daily_limit,
        "daily_ok": daily < self.daily_limit,
        "total_spend": total,
        "monthly_limit": self.monthly_limit,
        "monthly_ok": total < self.monthly_limit,
        "can_proceed": daily < self.daily_limit and total < self.monthly_limit,
    }

def get_breakdown(self) -> dict[str, float]:
    breakdown: dict[str, float] = defaultdict(float)
    for entry in self.ledger:
        breakdown[entry["operation"]] += entry["cost_usd"]
    return dict(breakdown)

CostTracker.get_daily_spend = get_daily_spend
CostTracker.get_total_spend = get_total_spend
CostTracker.check_limits = check_limits
CostTracker.get_breakdown = get_breakdown

print("✓ CostTracker defined")

### 分层记忆系统：把一切拼起来

`TieredMemorySystem` 是主协调器。它：
1. 在写入时扫描并脱敏 PII。
2. 通过 API 生成嵌入。
3. 把新记忆存入热层。
4. 跨所有层检索，并自动升级。
5. 定期维护：TTL 过期、预算限制、冷层降级。
6. 跟踪每次操作的成本。
7. 提供完整的用户删除（GDPR 被遗忘权）。


In [ ]:
class TieredMemorySystem:
    """带全部护栏的生产级分层记忆。

    协调热 / 温 / 冷三层、图存储、PII 处理、
    预算限制和成本跟踪。
    """

    def __init__(
        self,
        hot_max_per_user: int = 20,
        hot_ttl: int = 3600,
        budget_max_memories: int = 100,
        budget_max_tokens: int = 50_000,
        daily_cost_limit: float = 10.0,
    ):
        self.hot = HotTierCache(max_per_user=hot_max_per_user, default_ttl=hot_ttl)
        self.warm = WarmTierStore()
        self.cold = ColdTierArchive()
        self.graph = GraphRelationshipStore()
        self.pii = PIIHandler()
        self.budget = MemoryBudgetEnforcer(
            max_memories_per_user=budget_max_memories,
            max_tokens_per_user=budget_max_tokens,
        )
        self.cost = CostTracker(daily_limit=daily_cost_limit)

    def _generate_embedding(self, text: str) -> list[float]:
        """为演示生成确定性的伪嵌入。

        生产环境中请替换为：
            response = openai.embeddings.create(
                model="text-embedding-3-small", input=text
            )
            return response.data[0].embedding
        """
        # 基于哈希的确定性伪嵌入（保证演示可复现）
        h = hashlib.sha256(text.encode()).digest()
        rng = np.random.RandomState(int.from_bytes(h[:4], "big"))
        emb = rng.randn(256).tolist()
        # 按调用了真实 API 来记账
        tokens = estimate_tokens(text)
        self.cost.record("embedding", tokens, f"embed: {text[:40]}...")
        return emb

`add_memory` 方法跑完整个写入流程：扫描 PII、脱敏敏感内容、生成嵌入、创建 `MemoryRecord`，并把记录同时存入热层和温层。如果热层满了，最久未使用的记录会被淘汰到温层。


In [ ]:
def add_memory(
    self,
    user_id: str,
    content: str,
    importance: float = 0.5,
    tags: list[str] | None = None,
    ttl_seconds: int | None = None,
    redact_pii: bool = True,
) -> MemoryRecord:
    """通过完整写入流程添加一条记忆。"""
    # 1. PII 扫描与脱敏
    if redact_pii:
        content, detected_pii = self.pii.redact(content)
        pii_redacted = bool(detected_pii)
    else:
        pii_redacted = False

    # 2. 生成嵌入
    embedding = self._generate_embedding(content)

    # 3. 创建记录
    record = MemoryRecord(
        id=hashlib.md5(f"{user_id}:{content}:{time.time()}".encode()).hexdigest()[:12],
        user_id=user_id,
        content=content,
        embedding=embedding,
        importance=importance,
        tags=tags or [],
        ttl_seconds=ttl_seconds,
        pii_redacted=pii_redacted,
        token_estimate=estimate_tokens(content),
    )

    # 4. 存入热层（可能触发 LRU 淘汰到温层）
    evicted = self.hot.put(record)
    if evicted:
        self.warm.put(evicted)

    # 5. 同时存入温层以持久化
    self.warm.put(record)

    return record

TieredMemorySystem.add_memory = add_memory

`retrieve` 方法跨三层搜索。它先查热层（最快），再查温层（向量搜索），最后查冷层（访问时解压）。频繁访问的记录从温层升级到热层，冷层记录一旦被触碰就升级到温层。


In [ ]:
def retrieve(
    self,
    user_id: str,
    query: str,
    k: int = 5,
) -> list[MemoryRecord]:
    """跨所有层检索记忆，并自动升级。"""
    query_embedding = self._generate_embedding(query)
    results = []
    seen_ids = set()

    # 1. 先查热层
    hot_records = self.hot.get_user_records(user_id)
    if hot_records:
        q = np.array(query_embedding)
        q_norm = np.linalg.norm(q)
        scored = []
        for r in hot_records:
            if r.embedding is not None:
                e = np.array(r.embedding)
                e_norm = np.linalg.norm(e)
                if e_norm > 0 and q_norm > 0:
                    sim = float(np.dot(q, e) / (q_norm * e_norm))
                    scored.append((sim, r))
        scored.sort(key=lambda x: x[0], reverse=True)
        for _, r in scored[:k]:
            results.append(r)
            seen_ids.add(r.id)

    # 2. 搜索温层
    warm_results = self.warm.search(user_id, query_embedding, k=k)
    for r in warm_results:
        if r.id not in seen_ids:
            results.append(r)
            seen_ids.add(r.id)
            # 访问频繁则升级到热层
            if r.access_count >= 3:
                evicted = self.hot.put(r)
                if evicted and evicted.id != r.id:
                    self.warm.put(evicted)

    # 3. 查冷层（仅在结果不足时）
    if len(results) < k:
        cold_records = self.cold.get_user_records(user_id)
        for r in cold_records:
            if r.id not in seen_ids:
                # 被访问时升级到温层
                self.cold.remove(r.id)
                r.embedding = self._generate_embedding(r.content)
                self.warm.put(r)
                results.append(r)
                seen_ids.add(r.id)
                if len(results) >= k:
                    break

    return results[:k]

TieredMemorySystem.retrieve = retrieve

这些方法处理图关系和定期维护。`run_maintenance` 执行预算限制，并把陈旧、低重要度的记录从温层降级到冷存储。


In [ ]:
def add_relationship(
    self, user_id: str, source: str, relation: str,
    target: str, memory_id: str | None = None,
):
    """向图存储添加一条实体关系。"""
    edge = GraphEdge(
        source=source, relation=relation, target=target,
        user_id=user_id, memory_id=memory_id,
    )
    self.graph.add_edge(edge)

def query_relationships(self, entity: str, relation: str | None = None) -> list[GraphEdge]:
    return self.graph.query(entity, relation)

def run_maintenance(self, user_id: str) -> dict:
    """执行维护：落实预算限制，把候选记录降级到冷层。"""
    # 1. 预算限制
    all_warm = self.warm.get_user_records(user_id)
    budget_check = self.budget.check_budget(all_warm)

    evicted_to_cold = []
    if not budget_check["within_budget"]:
        for record in budget_check["eviction_candidates"]:
            self.warm.remove(record.id)
            self.cold.archive(record)
            evicted_to_cold.append(record.id)

    # 2. 把陈旧的温层记录降级到冷层
    stale = self.warm.get_low_access_records(user_id, threshold_days=30)
    for record in stale:
        if record.importance < 0.3:  # 只降级低重要度记录
            self.warm.remove(record.id)
            self.cold.archive(record)
            evicted_to_cold.append(record.id)

    return {
        "budget_check": budget_check,
        "demoted_to_cold": evicted_to_cold,
        "hot_count": len(self.hot.get_user_records(user_id)),
        "warm_count": len(self.warm.get_user_records(user_id)),
        "cold_count": len(self.cold.get_user_records(user_id)),
    }

TieredMemorySystem.add_relationship = add_relationship
TieredMemorySystem.query_relationships = query_relationships
TieredMemorySystem.run_maintenance = run_maintenance

`delete_user` 方法实现 GDPR 被遗忘权。它删除一个用户在所有层（热、温、冷和图）的全部数据。每次删除都会写入 PII 审计日志，留作合规凭证。`get_stats` 返回用于监控的可观测性指标。


In [ ]:
def delete_user(self, user_id: str) -> dict:
    """GDPR 被遗忘权：删除用户在所有层的全部数据。"""
    hot_deleted = self.hot.delete_user(user_id)
    warm_deleted = self.warm.delete_user(user_id)
    cold_deleted = self.cold.delete_user(user_id)
    graph_deleted = self.graph.delete_user(user_id)

    # 逐层写入审计日志
    for tier, count in [("hot", hot_deleted), ("warm", warm_deleted),
                        ("cold", cold_deleted), ("graph", graph_deleted)]:
        self.pii.log_deletion(user_id, tier, count)

    return {
        "user_id": user_id,
        "hot_deleted": hot_deleted,
        "warm_deleted": warm_deleted,
        "cold_deleted": cold_deleted,
        "graph_deleted": graph_deleted,
        "total_deleted": hot_deleted + warm_deleted + cold_deleted + graph_deleted,
        "audit_entries": len(self.pii.audit_log),
    }

def get_stats(self) -> dict:
    """获取系统级可观测性指标。"""
    return {
        "tiers": {
            "hot": {"count": len(self.hot), "hit_rate": f"{self.hot.hit_rate:.1%}",
                     "stats": self.hot.stats},
            "warm": {"count": len(self.warm)},
            "cold": {"count": len(self.cold), "bytes_saved": self.cold.bytes_saved},
        },
        "graph": {"edges": len(self.graph)},
        "cost": self.cost.check_limits(),
        "cost_breakdown": self.cost.get_breakdown(),
    }

TieredMemorySystem.delete_user = delete_user
TieredMemorySystem.get_stats = get_stats

print("✓ TieredMemorySystem defined - full production orchestrator")

## 用法示例：完整的生产流水线

我们走一遍完整生命周期：添加记忆（带 PII 脱敏）、跨层检索、查询图、执行预算、跟踪成本，最后删除一个用户的全部数据。


In [ ]:
# ── 初始化系统 ──
system = TieredMemorySystem(
    hot_max_per_user=5,       # 演示用的小缓存
    hot_ttl=3600,
    budget_max_memories=15,    # 演示用的紧预算
    budget_max_tokens=5_000,
    daily_cost_limit=10.0,
)

# ── 为用户 "alice" 添加记忆 ──
memories_data = [
    ("Alice works at Anthropic as a senior engineer.", 0.9, ["work"]),
    ("Alice lives in San Francisco, CA 94103.", 0.7, ["location"]),
    ("Alice email is alice@anthropic.com and phone is (415) 555-0123.", 0.6, ["contact"]),
    ("Alice prefers Python for data analysis and Rust for systems work.", 0.5, ["programming"]),
    ("Alice is training for the SF marathon in October 2025.", 0.4, ["fitness"]),
    ("Alice has a golden retriever named Max.", 0.3, ["pets"]),
    ("SSN is 123-45-6789 shared for benefits enrollment.", 0.8, ["admin"]),
    ("Alice enjoys hiking in Marin County on weekends.", 0.3, ["hobbies"]),
    ("Credit card 4111-1111-1111-1111 is on file for expenses.", 0.7, ["finance"]),
    ("Alice mentors two junior engineers on the safety team.", 0.5, ["work"]),
]

print("── Adding Memories (with PII Redaction) ──\n")
records = []
for content, importance, tags in memories_data:
    record = system.add_memory("alice", content, importance=importance, tags=tags)
    records.append(record)
    pii_flag = " ⚠ PII redacted" if record.pii_redacted else ""
    short = record.content[:65] + "..." if len(record.content) > 65 else record.content
    print(f"  [{record.tier.value:4s}] {record.id}: {short}{pii_flag}")

# ── 添加图关系 ──
system.add_relationship("alice", "Alice", "works_at", "Anthropic", records[0].id)
system.add_relationship("alice", "Alice", "lives_in", "San Francisco", records[1].id)
system.add_relationship("alice", "Alice", "owns_pet", "Max (golden retriever)", records[5].id)
system.add_relationship("alice", "Alice", "mentors", "junior engineers", records[9].id)
system.add_relationship("alice", "Anthropic", "located_in", "San Francisco")

print(f"\n✓ Added {len(records)} memories and {len(system.graph)} graph edges")

现在我们用自然语言查询检索记忆。系统跨所有层搜索，返回最相关的记录。留意输出里的层标签和访问次数。


In [ ]:
# ── 按语义查询检索记忆 ──
print("── Memory Retrieval ──\n")

queries = [
    "What does Alice do for work?",
    "Where does Alice live?",
    "What programming languages does Alice use?",
]

for query in queries:
    results = system.retrieve("alice", query, k=3)
    print(f"  Query: \"{query}\"")
    for r in results:
        short = r.content[:70]
        print(f"    → [{r.tier.value}] (importance={r.importance:.1f}, accesses={r.access_count}) {short}")
    print()

# ── 图关系查询 ──
print("── Graph Queries ──\n")

for entity in ["Alice", "Anthropic"]:
    edges = system.query_relationships(entity)
    print(f"  {entity}:")
    for e in edges:
        print(f"    → {e.relation} → {e.target}")

# 展示图遍历
neighbors = system.graph.get_neighbors("alice", depth=2)
print(f"\n  Entities within 2 hops of Alice: {neighbors}")

我们运行维护来执行预算限制，并降级陈旧记录。系统检查每用户的记忆条数和 token 限额。超出预算的记录会被移入冷存储。


In [ ]:
# ── 执行维护（预算限制 + 冷层降级） ──
print("── Maintenance & Budget Enforcement ──\n")

maintenance = system.run_maintenance("alice")

print(f"  Budget within limits: {maintenance['budget_check']['within_budget']}")
print(f"  Current count: {maintenance['budget_check']['current_count']} / {system.budget.max_memories}")
print(f"  Current tokens: {maintenance['budget_check']['current_tokens']} / {system.budget.max_tokens}")
if maintenance["demoted_to_cold"]:
    print(f"  Demoted to cold: {maintenance['demoted_to_cold']}")
print(f"\n  Tier distribution:")
print(f"    Hot:  {maintenance['hot_count']}")
print(f"    Warm: {maintenance['warm_count']}")
print(f"    Cold: {maintenance['cold_count']}")

# ── 成本跟踪 ──
print("\n── Cost Tracking ──\n")
stats = system.get_stats()
cost_info = stats["cost"]
print(f"  Daily spend:   ${cost_info['daily_spend']:.6f} / ${cost_info['daily_limit']:.2f}")
print(f"  Total spend:   ${cost_info['total_spend']:.6f} / ${cost_info['monthly_limit']:.2f}")
print(f"  Within limits: {cost_info['can_proceed']}")

print(f"\n  Cost breakdown by operation:")
for op, amount in stats["cost_breakdown"].items():
    print(f"    {op}: ${amount:.6f}")

print(f"\n  Cache performance:")
hot_stats = stats["tiers"]["hot"]
print(f"    Hit rate: {hot_stats['hit_rate']}")
print(f"    Hits: {hot_stats['stats']['hits']}, Misses: {hot_stats['stats']['misses']}")
print(f"    Evictions: {hot_stats['stats']['evictions']}")

最后我们测试 GDPR 被遗忘权。调用 `delete_user` 会把 Alice 的所有数据从每一层和图存储中清除。审计日志记录每次删除，作为合规凭证。


In [ ]:
# ── GDPR 被遗忘权：彻底删除用户数据 ──
print("── GDPR User Deletion ──\n")

# 展示删除前的状态
print("  Before deletion:")
print(f"    Hot tier:  {len(system.hot.get_user_records('alice'))} records")
print(f"    Warm tier: {len(system.warm.get_user_records('alice'))} records")
print(f"    Cold tier: {len(system.cold.get_user_records('alice'))} records")
print(f"    Graph:     {len(system.graph.edges)} edges")

# 执行删除
deletion_report = system.delete_user("alice")

print(f"\n  Deletion complete:")
print(f"    Hot deleted:   {deletion_report['hot_deleted']}")
print(f"    Warm deleted:  {deletion_report['warm_deleted']}")
print(f"    Cold deleted:  {deletion_report['cold_deleted']}")
print(f"    Graph deleted: {deletion_report['graph_deleted']}")
print(f"    Total deleted: {deletion_report['total_deleted']}")

# 验证删除是否彻底
print(f"\n  After deletion:")
print(f"    Hot tier:  {len(system.hot.get_user_records('alice'))} records")
print(f"    Warm tier: {len(system.warm.get_user_records('alice'))} records")
print(f"    Cold tier: {len(system.cold.get_user_records('alice'))} records")
print(f"    Graph:     {len(system.graph.edges)} edges")

# 展示审计日志
print(f"\n  Audit log ({len(system.pii.audit_log)} entries):")
for entry in system.pii.audit_log:
    print(f"    [{entry['timestamp']}] {entry['tier']}: {entry['records_deleted']} records deleted")

### 附加内容：带成本跟踪的 LLM 事实抽取

生产环境中，原始对话轮次会先交给 LLM 抽取结构化事实和关系，然后才入库。本单元格用真实 API 调用和成本跟踪演示这一模式。


In [ ]:
def extract_facts_and_relations(
    text: str,
    user_id: str,
    system: TieredMemorySystem,
) -> dict:
    """用 Claude 从文本中抽取结构化事实和关系。"""

    # 记录抽取成本
    input_tokens = estimate_tokens(text) + 200  # 提示词开销
    system.cost.record("extraction_input", input_tokens, "fact extraction")

    response = llm.invoke([
        {"role": "system", "content": (
            "Extract facts and entity relationships from the text. "
            "Return JSON with: "
            '"facts": [{"content": "...", "importance": 0.0-1.0, "tags": [...]}], '
            '"relationships": [{"source": "...", "relation": "...", "target": "..."}]'
        )},
        {"role": "user", "content": f"Extract from: {text}"},
    ])

    output_text = to_text(response.content).strip()
    system.cost.record("extraction_output", estimate_tokens(output_text), "fact extraction")

    try:
        if output_text.startswith("```"):
            output_text = output_text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        parsed = json.loads(output_text)
    except json.JSONDecodeError:
        parsed = {"facts": [], "relationships": []}

    # 存储抽取出的事实
    added_records = []
    for fact in parsed.get("facts", []):
        record = system.add_memory(
            user_id=user_id,
            content=fact["content"],
            importance=fact.get("importance", 0.5),
            tags=fact.get("tags", []),
        )
        added_records.append(record)

    # 存储关系
    for rel in parsed.get("relationships", []):
        system.add_relationship(
            user_id=user_id,
            source=rel["source"],
            relation=rel["relation"],
            target=rel["target"],
        )

    return {
        "facts_extracted": len(added_records),
        "relationships_extracted": len(parsed.get("relationships", [])),
        "records": added_records,
        "cost_so_far": system.cost.get_total_spend(),
    }

我们用一段真实感的对话片段测试抽取流程。用户提到了新住址、新工作、新经理和一只新宠物。观察抽取器如何从非结构化文本中抽出结构化事实和关系。注意 PII（街道地址）被自动脱敏了。


In [ ]:
print("── LLM Fact Extraction ──\n")

conversation = (
    "User: I just moved to Austin, Texas last week. My new address is "
    "742 Evergreen Terrace, Austin TX 78701. I am starting a new role as "
    "a tech lead at Stripe next Monday. My manager will be Sarah Chen. "
    "Oh and I adopted a cat named Pixel from the shelter!"
)

print(f"  Input: {conversation[:80]}...\n")

result = extract_facts_and_relations(conversation, "bob", system)

print(f"  Facts extracted: {result['facts_extracted']}")
print(f"  Relationships:   {result['relationships_extracted']}")
print(f"  Total cost:      ${result['cost_so_far']:.6f}")
print()

for r in result["records"]:
    pii_flag = " ⚠ PII" if r.pii_redacted else ""
    print(f"  → {r.content[:70]}{pii_flag}")

print(f"\n  Graph edges for Bob's entities:")
for entity in ["bob", "stripe", "sarah chen", "austin", "pixel"]:
    edges = system.graph.query(entity)
    for e in edges:
        print(f"    {e.source} → {e.relation} → {e.target}")

## 讨论与权衡

### 为什么用分层存储？

| 层 | 底层存储 | 延迟 | 成本/GB/月 | 使用场景 |
|------|--------------|---------|------------|----------|
| **热** | Redis / Memcached | <1 ms | ~$25 | 当前会话上下文、高频查询 |
| **温** | Postgres + pgvector | 5-50 ms | ~$0.25 | 长期事实、语义搜索 |
| **冷** | S3 / Glacier / 压缩 | 100 ms+ | ~$0.004 | 归档、低频访问、合规留存 |

热存储和冷存储之间 100 倍的成本差，让分层在规模化后成为必需。一个存了一万条记忆的用户，全放温层每月花 $0.25；全缓存在 Redis 里则每月花 $25。

### 实践中的 PII 处理

上面这种基于正则的 PII 检测能抓住常见模式，但有明显局限：
- **漏报：** 没有 NER（命名实体识别）时，它检测不出姓名、地址或依赖上下文的 PII。
- **误报：** 它可能把合法数据也脱敏掉（比如技术日志里的 IP 地址）。
- **嵌入泄漏：** 即使文本已脱敏，嵌入仍可能编码了 PII。可以考虑训练 PII 感知的嵌入模型，或使用差分隐私。

**生产建议：** 用 Microsoft Presidio 或云端 DLP 服务做检测。PII 字段静态加密。另建一张可独立删除的 PII 映射表。

### 预算限制策略

这里展示的重要性加权淘汰只是其中一种做法。其他可选方案：

| 策略 | 优点 | 缺点 |
|----------|------|------|
| **LRU（最近最少使用）** | 无需打分 | 可能淘汰重要但不常访问的记忆 |
| **重要性加权**（本 notebook） | 保住高价值记忆 | 依赖准确的重要性分数 |
| **仅用 TTL** | 行为可预期，无需决策逻辑 | 时间并不总是正确的信号 |
| **token 预算加摘要** | 以浓缩形式保留全部信息 | 摘要有成本，且可能丢失细节 |

### 成本控制的经验

1. **批量生成嵌入。** 一条一条生成嵌入比批量贵 3-5 倍。先积累记忆，再批量嵌入。
2. **缓存嵌入。** 同一段文本嵌入两次就是浪费钱。对内容做哈希，缓存嵌入结果。
3. **用小模型做抽取。** Haiku 级模型做结构化事实抽取效果不错，成本只有 Opus 的十分之一到二十分之一。
4. **设硬性每日限额。** 抽取循环里的一个 bug 能在几小时内烧光你的月度预算。硬性每日上限必不可少。
5. **监控每用户成本。** 有些用户产生的记忆操作是别人的 100 倍。按用户跟踪成本，避免交叉补贴问题。

### 本 notebook 未覆盖的内容

- **水平分片。** 按用户 ID 哈希把数据分散到多个节点。用户超过 10 万时至关重要，但会增加复杂度（跨分片查询、再均衡）。
- **分布式缓存。** 用 Redis Cluster 或 Memcached 保持多节点缓存一致。
- **实时可观测性。** OpenTelemetry 埋点、Prometheus 指标、Grafana 仪表盘。
- **备份与恢复。** WAL 归档、时间点恢复、跨区域复制。
- **记忆系统 A/B 测试。** 并行运行两套记忆配置，比较下游任务质量。


## 延伸阅读

- [Redis 文档：数据持久化与 TTL](https://redis.io/docs/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 生产级缓存模式、TTL 配置和淘汰策略。
- [pgvector：Postgres 的开源向量相似度搜索](https://github.com/pgvector/pgvector) - 嵌入存储、索引（IVFFlat、HNSW）和查询优化。
- [GDPR 第 17 条：删除权（被遗忘权）](https://gdpr-info.eu/art-17-gdpr/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 数据删除的法律要求，包括嵌入这类派生数据。
- [Microsoft Presidio：PII 检测与匿名化](https://github.com/microsoft/presidio) - 检测和脱敏文本、图像与结构化数据中 PII 的开源框架。
- [OpenTelemetry：可观测性框架](https://opentelemetry.io/docs/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 生产系统的埋点、追踪和指标。
- [Neo4j：面向知识图谱的图数据库](https://neo4j.com/docs/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 生产级图存储，支持实体关系、遍历查询和知识表示。
- [Pinecone：生产级向量数据库最佳实践](https://docs.pinecone.io/guides/getting-started/overview?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 托管向量数据库的扩展、分片和成本优化。
- [Anthropic：构建高效智能体](https://www.anthropic.com/engineering/building-effective-agents?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 生产级智能体架构模式，包括记忆管理。

---

*← 上一篇：[29 - 记忆基准测试（LoCoMo）](../29_memory_benchmarks_LoCoMo/) →*


## 🧪 动手练习

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：自定义 PII 模式
为 `PIIHandler` 添加一个新的银行卡号正则模式。构造几条嵌有卡号的对话文本作为测试用例。验证 `PIIHandler` 能检测并脱敏它们，同时保留消息其余部分。检查新模式的审计日志。

### 挑战 2：层间迁移跟踪
向 `TieredMemorySystem` 添加 30 条记忆，运行 5 次 `run_maintenance()`。每轮之后，用 10 个查询调用 `retrieve()` 生成访问模式，然后再跑一次维护。统计有多少记忆在热、温、冷三层之间迁移。画出每轮维护后各层的记录数。

### 挑战 3：预算受限下的淘汰
在 `MemoryBudgetEnforcer` 里设一个很紧的每用户上限（比如 50 条记忆）。添加 100 条重要性各异的记忆。触发淘汰，验证被淘汰的是重要性最低的记忆。比较存活记忆与被淘汰记忆的平均重要性。这会用到 19 遗忘与衰减里的衰减和修剪策略。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--30-production-memory-patterns--production-memory-patterns)
